In [ ]:
import sys
sys.path.append('../')
import torch
import os
import yaml
from tqdm import tqdm
from torch.utils.data import DataLoader
from dataset import GraphDataset
from models.vae import GeoVAE

In [ ]:
config_path = '../configs/latent_diffusion.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    
device = torch.device(config['device'])
print(f"Using device: {device}")

In [ ]:
print("Loading VAE...")
vae_config = config['vae_config']
vae = GeoVAE(vae_config['model_config']).to(device)
    
# 加载权重
ckpt = torch.load(vae_config['ckpt_path'], map_location=device)
if 'model_state_dict' in ckpt:
    vae.load_state_dict(ckpt['model_state_dict'])
else:
    vae.load_state_dict(ckpt)
        
vae.eval()

In [ ]:
# 3. 加载训练数据
print("Loading Training Dataset...")
data_config = config['data_config']
train_dataset = GraphDataset(
    data_path=data_config['train_data_path'],
    data_config=data_config
)
    
# 必须使用 shuffle=False，虽然对统计没影响，但保持确定性好习惯
loader = DataLoader(
    train_dataset, 
    batch_size=1, # Batch size 可以大一点，只做前向
    shuffle=False, 
    collate_fn=train_dataset.collate_fn,
    num_workers=4
)

In [ ]:
# 4. 收集所有 Latent Vectors
print("Extracting latents...")
all_mus = []
    
with torch.no_grad():
    for batch in tqdm(loader, desc="Scanning Dataset"):
        batch = batch.to(device)
        
        # 我们只需要 mu 来计算数据集的中心和分布范围
        mu, _, mask = vae.encode(batch) # mu: [B, L, D]
            
        # Mask out padding (只保留真实节点)
        mask_bool = mask.bool()
        valid_mu = mu[mask_bool] # [N_valid, D]
            
        all_mus.append(valid_mu.cpu()) # 放到 CPU 防止显存溢出

In [ ]:
# 5. 拼接并计算统计量
print("Calculating statistics...")
all_mus = torch.cat(all_mus, dim=0) # [Total_Nodes, D]
print(f"Total valid nodes: {all_mus.shape[0]}")

latent_mean = torch.mean(all_mus, dim=0)
latent_std = torch.std(all_mus, dim=0)

save_dir = os.path.dirname(vae_config['ckpt_path'])
save_path = os.path.join(save_dir, 'vae_latent_stats.pt')
    
torch.save({
    'mean': latent_mean,
    'std': latent_std
}, save_path)

print(f"Statistics saved to: {save_path}")